# ☁️ HollyShit 五师特征矩阵云对云迁移 (`ModelScope -> Hugging Face Hub`)

本 Notebook 将魔搭社区公开数据集 **`catclever/emb_npy`** 的全部 18 个对齐宏分块（`bge-m3` / `gte_Qwen2-7B` / `xiaobu-v2` / `Youtu` / `Conan-v1`，共 90 个 `.npz` 文件，约 `~140 GB`）流式直传至你的 **Hugging Face 公开数据集 `<your_hf_username>/emb_npy`**，彻底解决海外 Colab 实例拉取跨境大文件断流慢速的问题。

### 🚀 三步极简启动指南
1. **选择免费 CPU 运行时**：点击顶部菜单 `代码执行程序 -> 更改运行时类型`，硬件加速器选择 **`CPU` / `None`**（全程纯网络 I/O 流转，**零 GPU 计算单元消耗**）。
2. **配置 `HF_TOKEN` 密钥**：点击左侧栏 🔑 **Secrets (密钥)** 图标，新增名称为 `HF_TOKEN`、权限为 **Write (写入)** 的 Hugging Face Access Token，并打开右侧「Notebook 访问权限」开关。
3. **点击「全部运行 (Run All)」**：脚本将自动创建公开数据集 `<your_hf_username>/emb_npy`，按宏分块逐块下载 -> 零内存 `.npz` 头部与 EOF 完整性校验 -> 单次原子提交至 HF -> 立即清空本地暂存，**`/content` 峰值磁盘占用 `< 9 GB`，且支持任意断线重连秒级断点续传**！

In [ ]:
import os
import shutil

# 1. 安装 Hugging Face 高速传输引擎 (hf_transfer) 与魔搭 SDK
!pip install -q "huggingface_hub[hf_transfer]" modelscope pyarrow || pip install -q huggingface_hub modelscope pyarrow

try:
    import hf_transfer  # noqa: F401
    os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
    print('⚡ 已启用 hf_transfer Rust 多线程高速传输加速 (HF_HUB_ENABLE_HF_TRANSFER=1)')
except ImportError:
    os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
    print('ℹ️ 当前环境未安装 hf_transfer 轮子，已安全回退至标准 HTTP 传输模式')

# 2. 从 Colab Secrets 读取 HF_TOKEN (必需，需具备 Write 权限) 与可选 MODELSCOPE_API_TOKEN
hf_tok = os.environ.get('HF_TOKEN', '').strip()
try:
    from google.colab import userdata
    if not hf_tok:
        secret_hf = userdata.get('HF_TOKEN')
        if secret_hf:
            hf_tok = secret_hf.strip()
except Exception:
    pass

if not hf_tok:
    raise RuntimeError(
        '❌ 未在 Colab Secrets 中检测到 HF_TOKEN！\n'
        '👉 请点击左侧栏 🔑 (Secrets) 图标，添加名为 HF_TOKEN (需具备 Write 写入权限) 的密钥，'
        '打开 Notebook 访问开关后重新运行本单元格！'
    )
os.environ['HF_TOKEN'] = hf_tok

try:
    from google.colab import userdata
    ms_tok = userdata.get('MODELSCOPE_API_TOKEN')
    if ms_tok:
        os.environ['MODELSCOPE_API_TOKEN'] = ms_tok.strip()
        print('🔑 已加载可选 MODELSCOPE_API_TOKEN')
except Exception:
    pass

# 3. 可选挂载 Google Drive (仅当希望顺带把云盘内的 chunked_mixed_omni.parquet 一并推上 HF 时设为 True；默认 False 免弹窗阻塞)
MOUNT_DRIVE_FOR_PARQUET = False
if MOUNT_DRIVE_FOR_PARQUET and not os.path.ismount('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    assert os.path.ismount('/content/drive'), '❌ Google Drive 挂载未完成！'

# 4. 拉取或更新 HollyShit 仓库代码 (含词表 char_vocab.json 与迁移脚本)
if not os.path.exists('/content/HollyShit'):
    !git clone https://github.com/catclever/HollyShit.git /content/HollyShit
    %cd /content/HollyShit
    !git submodule update --init training/core
else:
    %cd /content/HollyShit
    !git pull
    !git submodule update --init training/core

# 5. 校验 /content 可用磁盘空间与解析目标 Hugging Face 数据集仓库
from huggingface_hub import HfApi

hf_user = HfApi(token=hf_tok).whoami()['name']
target_hf_repo = f'{hf_user}/emb_npy'
free_gb = shutil.disk_usage('/content').free / (1024.0 ** 3)

print('\n' + '=' * 68)
print(f'👤 Hugging Face 账号 : {hf_user}')
print(f'🎯 目标公开数据集    : https://huggingface.co/datasets/{target_hf_repo}')
print(f'💾 /content 可用空间 : {free_gb:.1f} GB (单块滚动暂存峰值仅需 ~8.5 GB)')
print('=' * 68)
assert free_gb >= 12.0, f'❌ /content 可用空间不足 ({free_gb:.1f} GB < 12 GB)，请重置运行时清理临时空间！'


In [ ]:
# 启动云对云可断点续传流式迁移 (若中途断线，重新点击 Run All 即可秒级跳过已传分块继续迁移)
%cd /content/HollyShit
!python -u scripts/migrate_modelscope_to_hf.py \
  --ms_repo_id catclever/emb_npy \
  --work_dir /content/ms_to_hf_staging \
  --parquet_path /content/drive/MyDrive/hollyshit/data/Basic_ZH/chunked_mixed_omni.parquet \
  --vocab_path /content/HollyShit/training/core/char_vocab.json
